### **AutoLoader**

In [0]:
import os
import sys

project_path=os.path.join(os.getcwd(),'..','..')
sys.path.append(project_path)

In [0]:
from utils.transformations import *

In [0]:
project_path

In [0]:
df=spark.read.format('parquet')\
             .load('abfss://bronze@storageaccntazureproject.dfs.core.windows.net/DimUser')

In [0]:
df.show()

In [0]:
display(df)

####**DimUser**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:

df_user = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.includeExistingFiles", "true") \
    .option("rescuedDataColumn", "_rescued_data") \
    .option(
        "cloudFiles.schemaLocation",
        "abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimUser/checkpoint"
    ) \
    .load(
        "abfss://bronze@storageaccntazureproject.dfs.core.windows.net/DimUser"
    )


In [0]:
df_user = df_user.withColumn(
    "user_name",
    upper(col("user_name"))
)


In [0]:
df_user.printSchema()

In [0]:
from utils.transformations import reusable

In [0]:
db_user_obj=reusable()
df_user=db_user_obj.dropColumns(df_user,['_rescued_data'])
df_user=df_user.dropDuplicates(['user_id'])
# display(df_user,checkpointLocation="abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimUser/checkpoint")

In [0]:
df_user.printSchema()

In [0]:
# dbutils.fs.rm(
#     "abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimUser/checkpoint",
#     True
# )

In [0]:
df_user.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimUser/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimUser/data")\
    .toTable("spotify_cata.silver.dimuser")

In [0]:
%sql
select * from spotify_cata.silver.dimuser

### **DimArtist**

In [0]:
from pyspark.sql.functions import *

df_art = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("rescuedDataColumn", "_rescued_data") \
    .option(
        "cloudFiles.schemaLocation",
        "abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimArtist/checkpoint"
    ) \
    .load(
        "abfss://bronze@storageaccntazureproject.dfs.core.windows.net/DimArtist"
    )

In [0]:
# display(
#     df_art,
#     checkpointLocation="abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimArtist/checkpoint"
# )

In [0]:
df_art.printSchema()

In [0]:
db_art_obj=reusable()
df_art=db_art_obj.dropColumns(df_art,['_rescued_data'])
df_art=df_art.dropDuplicates(['artist_id'])

In [0]:
df_art = df_art.withColumn(
    "artist_name",
    upper(col("artist_name"))
)

In [0]:
df_art.printSchema()

In [0]:
# dbutils.fs.rm(
#     "abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimArtist/checkpoint",
#     True
# )

In [0]:
df_art.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimArtist/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimArtist/data")\
    .toTable("spotify_cata.silver.dimartist")

In [0]:
%sql
select * from spotify_cata.silver.dimartist

### **DimTrack**

In [0]:
from pyspark.sql.functions import *

df_track = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("rescuedDataColumn", "_rescued_data") \
    .option(
        "cloudFiles.schemaLocation",
        "abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimTrack/checkpoint"
    ) \
    .load(
        "abfss://bronze@storageaccntazureproject.dfs.core.windows.net/DimTrack"
    )

In [0]:
# display(
#     df_track,
#     checkpointLocation="abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimTrack/checkpoint"
# )

In [0]:
df_track.printSchema()

In [0]:
db_track_obj=reusable()
df_track=db_track_obj.dropColumns(df_track,['_rescued_data'])
df_track=df_track.dropDuplicates(['track_id'])

In [0]:
df_track = df_track.withColumn(
    "track_name",
    upper(col("track_name"))
)

In [0]:
df_track=df_track.withColumn("durationFlag",when(col('duration_sec')<150,"low")\
                                            .when(col('duration_sec')<300,"medium")\
                                            .otherwise("high"))

**Replacing '-' with space(')**

In [0]:
df_track=df_track.withColumn("track_name",regexp_replace(col("track_name"),'-',' '))

In [0]:
df_track.printSchema()

In [0]:
# dbutils.fs.rm(
#     "abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimTrack/checkpoint",
#     True
# )

In [0]:
df_track.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimTrack/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimTrack/data")\
    .toTable("spotify_cata.silver.dimtrack")

In [0]:
%sql
select * from spotify_cata.silver.dimtrack

### **DimDate**

In [0]:
from pyspark.sql.functions import *

df_date = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("rescuedDataColumn", "_rescued_data") \
    .option(
        "cloudFiles.schemaLocation",
        "abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimDate/checkpoint"
    ) \
    .load(
        "abfss://bronze@storageaccntazureproject.dfs.core.windows.net/DimDate"
    )

In [0]:
# display(
#     df_date,
#     checkpointLocation="abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimDate/checkpoint"
# )

In [0]:
df_date.printSchema()

In [0]:
db_date_obj=reusable()
df_date=db_date_obj.dropColumns(df_date,['_rescued_data'])

In [0]:
df_date.printSchema()

In [0]:
df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimDate/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageaccntazureproject.dfs.core.windows.net/DimDate/data")\
    .toTable("spotify_cata.silver.dimdate")

In [0]:
%sql
select * from spotify_cata.silver.dimdate

### **FactStream**

In [0]:
from pyspark.sql.functions import *

df_fact = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("rescuedDataColumn", "_rescued_data") \
    .option(
        "cloudFiles.schemaLocation",
        "abfss://silver@storageaccntazureproject.dfs.core.windows.net/FactStream/checkpoint"
    ) \
    .load(
        "abfss://bronze@storageaccntazureproject.dfs.core.windows.net/FactStream"
    )

In [0]:
df_fact.printSchema()

In [0]:
db_fact_obj=reusable()
df_fact=db_fact_obj.dropColumns(df_fact,['_rescued_data'])

In [0]:
df_fact.printSchema()

In [0]:
df_fact.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@storageaccntazureproject.dfs.core.windows.net/FactStream/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageaccntazureproject.dfs.core.windows.net/FactStream/data")\
    .toTable("spotify_cata.silver.factstream")

In [0]:
%sql
select * from spotify_cata.silver.factstream